In [1]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset

IMG_SIZE = 512   # table nên >=256, 512 là ổn

class TableSpaceDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir

        self.files = sorted([
            f for f in os.listdir(img_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]

        img_path  = os.path.join(self.img_dir, fname)
        mask_path = os.path.join(self.mask_dir, fname)

        # ---------- IMAGE ----------
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # ---------- MASK ----------
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # ---------- RESIZE ----------
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE),
                         interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE),
                          interpolation=cv2.INTER_NEAREST)

        # ---------- NORMALIZE ----------
        img = img.astype(np.float32) / 255.0
        mask = (mask > 0).astype(np.float32)

        # ---------- TO TENSOR ----------
        img = img.transpose(2, 0, 1)     # (3, H, W)
        mask = mask[None, :, :]          # (1, H, W)

        return (
            torch.tensor(img, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32)
        )


In [4]:
import segmentation_models_pytorch as smp

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(DEVICE)

dice_loss = smp.losses.DiceLoss(mode="binary")
bce_loss  = smp.losses.SoftBCEWithLogitsLoss()

def loss_fn(pred, target):
    return dice_loss(pred, target) + bce_loss(pred, target)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
from torch.utils.data import DataLoader

train_dataset = TableSpaceDataset(
    img_dir=r"D:\USTH\Computer_Vision\table\crop_tables",
    mask_dir=r"D:\USTH\Computer_Vision\table\row_space_masks"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,      # table to, batch nhỏ thôi
    shuffle=True,
    num_workers=4,
    pin_memory=True
)


In [ ]:
from tqdm.notebook import tqdm
import torch
import os
from torch import optim
from tqdm import tqdm

SAVE_DIR = "checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_PATH = os.path.join(SAVE_DIR, "best.pth")
LAST_PATH = os.path.join(SAVE_DIR, "last.pth")

best_loss = float("inf")

model.train()

for epoch in range(30):
    total_loss = 0.0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1:02d}/30",
        leave=False
    )

    for imgs, masks in pbar:
        imgs  = imgs.to(DEVICE)
        masks = masks.to(DEVICE)

        preds = model(imgs)
        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # cập nhật loss realtime trên tqdm
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1:02d} | avg_loss = {avg_loss:.4f}")

    # ---------- SAVE LAST ----------
    torch.save({
        "epoch": epoch + 1,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "loss": avg_loss
    }, LAST_PATH)

    # ---------- SAVE BEST ----------
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            "epoch": epoch + 1,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "loss": best_loss
        }, BEST_PATH)

        print(f"  ⭐ New BEST model saved (loss={best_loss:.4f})")


Epoch 01/30:   0%|          | 0/5382 [00:00<?, ?it/s]